# 多模态评估指标入门教程

本教程介绍视觉-语言模型常用的评估指标。

## 学习目标

- 理解不同任务的评估方法
- 掌握 BLEU、ROUGE、CIDEr 等指标
- 学会使用评估工具

## 目录

1. [评估任务概览](#1-评估任务概览)
2. [图像描述评估](#2-图像描述评估)
3. [检索评估](#3-检索评估)
4. [VQA 评估](#4-vqa-评估)
5. [视觉定位评估](#5-视觉定位评估)
6. [实践练习](#6-实践练习)

In [ ]:
# 环境准备
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

from evaluation import (
    BLEU, ROUGE, CIDEr, VQAMetrics, 
    GroundingMetrics, RetrievalMetrics,
    ClassificationMetrics, MultimodalEvaluator
)

print('评估模块导入成功!')

## 1. 评估任务概览

| 任务 | 常用指标 | 说明 |
|------|----------|------|
| 图像描述 | BLEU, ROUGE, CIDEr | 生成文本与参考文本的相似度 |
| 图文检索 | Recall@K | 检索准确率 |
| 视觉问答 | VQA Accuracy | 答案准确率 |
| 视觉定位 | IoU | 边界框重叠度 |
| 零样本分类 | Top-1/5 Accuracy | 分类准确率 |

## 2. 图像描述评估

### 2.1 BLEU (Bilingual Evaluation Understudy)

计算生成文本与参考文本的 n-gram 重叠度。

In [ ]:
# BLEU 示例
candidate = ["a", "cat", "sitting", "on", "a", "mat"]
references = [
    ["a", "cat", "is", "sitting", "on", "the", "mat"],
    ["there", "is", "a", "cat", "on", "the", "mat"]
]

score = BLEU.compute_bleu(candidate, references, max_n=4)
print(f'BLEU-4 分数: {score:.4f}')

In [ ]:
# 不同 n-gram 的 BLEU
for n in [1, 2, 3, 4]:
    score = BLEU.compute_bleu(candidate, references, max_n=n)
    print(f'BLEU-{n}: {score:.4f}')

### 2.2 ROUGE-L

基于最长公共子序列 (LCS) 的评估指标。

In [ ]:
# ROUGE-L 示例
candidate = ["the", "cat", "sat", "on", "the", "mat"]
reference = ["the", "cat", "is", "sitting", "on", "the", "mat"]

result = ROUGE.compute_rouge_l(candidate, reference)
print(f'Precision: {result["precision"]:.4f}')
print(f'Recall: {result["recall"]:.4f}')
print(f'F1: {result["f1"]:.4f}')

### 2.3 CIDEr

基于 TF-IDF 的共识评分，更关注"独特"的词汇。

In [ ]:
# CIDEr 示例
cider = CIDEr(n=4)

candidates = [
    ["a", "dog", "running", "in", "the", "park"],
    ["a", "cat", "sleeping", "on", "a", "sofa"]
]
references_list = [
    [["a", "dog", "is", "running", "in", "the", "park"]],
    [["a", "cat", "is", "sleeping", "on", "the", "sofa"]]
]

results = cider.evaluate(candidates, references_list)
print(f'CIDEr 分数: {results["cider"].score:.4f}')

## 3. 检索评估

### Recall@K

前 K 个检索结果中包含正确答案的比例。

In [ ]:
# 模拟检索场景
num_samples = 100
embed_dim = 256

# 创建匹配的图像-文本特征
image_features = F.normalize(torch.randn(num_samples, embed_dim), dim=-1)
text_features = image_features + 0.1 * torch.randn_like(image_features)  # 添加噪声
text_features = F.normalize(text_features, dim=-1)

# 计算相似度
similarity = image_features @ text_features.T

# 评估
for k in [1, 5, 10]:
    recall = RetrievalMetrics.recall_at_k(similarity, k=k, direction='i2t')
    print(f'Recall@{k}: {recall:.4f}')

## 4. VQA 评估

### VQA Accuracy

标准 VQA 评估：`min(1, 答案出现次数 / 3)`

In [ ]:
# VQA 评估示例
predictions = ["cat", "blue", "three"]
ground_truths = [
    ["cat", "cat", "cat", "kitten"],  # 3/4 人说 cat
    ["blue", "blue", "navy"],          # 2/3 人说 blue
    ["3", "three", "3"]                # 需要标准化
]

results = VQAMetrics.evaluate(predictions, ground_truths)
print(f'VQA Accuracy: {results["vqa_accuracy"].score:.4f}')
print(f'Exact Match: {results["exact_match"].score:.4f}')

## 5. 视觉定位评估

### IoU (Intersection over Union)

预测框与真实框的重叠程度。

In [ ]:
# IoU 计算示例
pred_box = [10, 10, 50, 50]   # [x1, y1, x2, y2]
gt_box = [20, 20, 60, 60]

iou = GroundingMetrics.compute_iou(pred_box, gt_box)
print(f'IoU: {iou:.4f}')

In [ ]:
# 可视化 IoU
fig, ax = plt.subplots(figsize=(6, 6))

# 绘制边界框
pred_rect = plt.Rectangle((10, 10), 40, 40, fill=False, color='blue', linewidth=2, label='预测框')
gt_rect = plt.Rectangle((20, 20), 40, 40, fill=False, color='red', linewidth=2, label='真实框')
ax.add_patch(pred_rect)
ax.add_patch(gt_rect)

# 标注交集
inter_rect = plt.Rectangle((20, 20), 30, 30, alpha=0.3, color='green', label='交集')
ax.add_patch(inter_rect)

ax.set_xlim(0, 80)
ax.set_ylim(0, 80)
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'IoU = {iou:.4f}')
plt.show()

## 6. 实践练习

### 使用统一评估器

In [ ]:
# 创建统一评估器
evaluator = MultimodalEvaluator()

# 评估图像描述
candidates = [["a", "dog", "in", "the", "park"]]
references = [[["a", "dog", "playing", "in", "the", "park"]]]

caption_results = evaluator.evaluate_captioning(candidates, references)
for name, result in caption_results.items():
    print(f'{name}: {result.score:.4f}')

In [ ]:
# 评估分类
logits = torch.tensor([[0.1, 0.8, 0.1], [0.7, 0.2, 0.1], [0.2, 0.3, 0.5]])
labels = torch.tensor([1, 0, 2])

cls_results = evaluator.evaluate_classification(logits, labels)
for name, result in cls_results.items():
    print(f'{name}: {result.score:.4f}')

## 总结

### 指标选择指南

| 任务 | 推荐指标 | 原因 |
|------|----------|------|
| 图像描述 | CIDEr | 考虑词汇独特性 |
| 机器翻译 | BLEU | 标准评估指标 |
| 摘要生成 | ROUGE-L | 关注召回率 |
| VQA | VQA Accuracy | 标准 VQA 评估 |
| 目标检测 | mAP@IoU | 综合评估 |